# Fase 3 — Feature Engineering: Silver → Gold

Constrói o dataset Gold aplicando:
- Features temporais (hora, turno, dia da semana)
- Frequência de alarmes em janelas rolantes (30m / 1h / 4h)
- Aceleração de críticos (razão últia hora / média das 4h)
- Alarm fingerprint: presença dos top-30 alarmes na janela de 4h
- Contexto do equipamento (frota, estado do apontamento)

**Target:** `is_dont_go_next_60m` — ocorrerá um evento Don't Go nos próximos 60 minutos?

In [ ]:
import sys
sys.path.insert(0, '../src')

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from features import build_feature_matrix, get_feature_columns, ROLLING_WINDOWS, TOP_N_FINGERPRINT

SILVER_DIR = Path('../outputs/silver')
GOLD_DIR = Path('../outputs/gold')

print('Arquivos silver disponíveis:')
for f in sorted(SILVER_DIR.glob('*.parquet')):
    size_mb = f.stat().st_size / 1024**2
    print(f'  {f.name}  ({size_mb:.1f} MB)')

## 1. Construir o Gold Dataset

In [ ]:
expected_months = [f.stem.replace('silver_', '') for f in sorted(SILVER_DIR.glob('silver_*.parquet'))]
gold_files = [GOLD_DIR / f'gold_{m}.parquet' for m in expected_months]
gold_ready = all(f.exists() for f in gold_files)

if gold_ready:
    print(f'Gold já existe — carregando {len(gold_files)} arquivo(s)')
    df = pl.scan_parquet([str(f) for f in gold_files]).collect()
    top_alarm_ids = [int(c.split('_')[-1]) for c in df.columns if c.startswith('fp_alarm_')]
else:
    missing = [f.name for f in gold_files if not f.exists()]
    print(f'Gerando gold ({len(missing)} meses ausentes: {missing})')
    lf, top_alarm_ids = build_feature_matrix(save=True)
    df = lf.collect()

print(f'\nGold dataset: {len(df):,} registros × {len(df.columns)} colunas')
print(f'Fingerprint baseado em {len(top_alarm_ids)} alarmes')

## 2. Visão Geral do Dataset

In [ ]:
feature_cols = get_feature_columns(df)
print(f'Features para ML: {len(feature_cols)}')
print(f'\nColunas target disponíveis:')
for c in df.columns:
    if 'dont_go' in c or 'minutes_to' in c:
        print(f'  {c}: {df[c].dtype}')

In [ ]:
# Distribuição dos targets de janela
rows = []
for col in ['is_dont_go_next_60m', 'is_dont_go_next_120m', 'is_dont_go_next_240m']:
    positivos = df[col].sum()
    total = len(df)
    rows.append({'janela': col.replace('is_dont_go_next_', ''), 'positivos': positivos,
                 'total': total, 'pct': round(100 * positivos / total, 3)})

target_dist = pl.DataFrame(rows)
print(target_dist)

In [ ]:
fig = px.bar(
    target_dist.to_pandas(),
    x='janela', y='pct',
    title='% de registros positivos por janela de look-ahead',
    labels={'janela': 'Janela', 'pct': '% positivos'},
    text_auto='.3f',
    color='janela',
)
fig.update_traces(textposition='outside')
fig.show()

## 3. Análise das Features por Grupo

In [ ]:
# Frequência de alarmes: comparação positivos vs negativos
freq_cols = [f'n_alarmes_{w}m' for w in ROLLING_WINDOWS] + [f'n_criticos_{w}m' for w in ROLLING_WINDOWS]
freq_cols = [c for c in freq_cols if c in df.columns]

stats = (
    df.group_by('is_dont_go_next_60m')
      .agg([pl.col(c).mean().alias(c) for c in freq_cols])
      .sort('is_dont_go_next_60m')
)
print('Médias de frequência de alarmes por classe:')
print(stats)

In [ ]:
# Hora do dia: distribuição de DG por hora
dg_por_hora = (
    df.filter(pl.col('is_dont_go_next_60m'))
      .group_by('hora_dia')
      .agg(pl.len().alias('count'))
      .sort('hora_dia')
)

fig = px.bar(
    dg_por_hora.to_pandas(),
    x='hora_dia', y='count',
    title='Eventos pré-Don\'t Go por hora do dia (janela 60m)',
    labels={'hora_dia': 'Hora', 'count': 'Contagem'},
)
fig.show()

In [ ]:
# Aceleração de críticos
fig = px.histogram(
    df.filter(pl.col('aceleracao_criticos') < 20).to_pandas(),
    x='aceleracao_criticos',
    color='is_dont_go_next_60m',
    barmode='overlay',
    nbins=60,
    title='Aceleração de críticos: pré-DG vs normal',
    labels={'aceleracao_criticos': 'Aceleração (razão últia 1h / média 4h)', 'count': 'Freq'},
    opacity=0.6,
)
fig.show()

## 4. Alarm Fingerprint — Top Alarmes

In [ ]:
fp_cols = [c for c in df.columns if c.startswith('fp_alarm_')]

# Taxa de presença de cada alarme fingerprint em positivos vs negativos
fp_pos = df.filter(pl.col('is_dont_go_next_60m')).select(fp_cols).mean()
fp_neg = df.filter(~pl.col('is_dont_go_next_60m')).select(fp_cols).mean()

fp_df = pl.DataFrame({
    'alarme': fp_cols,
    'pct_positivo': fp_pos.row(0),
    'pct_negativo': fp_neg.row(0),
}).with_columns(
    (pl.col('pct_positivo') / (pl.col('pct_negativo') + 1e-6)).alias('lift')
).sort('lift', descending=True)

print('Top-10 alarmes fingerprint por lift:')
print(fp_df.head(10))

In [ ]:
fig = px.bar(
    fp_df.head(15).to_pandas(),
    x='alarme', y='lift',
    title='Top-15 alarmes fingerprint por lift (pré-DG vs normal)',
    labels={'alarme': 'Alarme ID', 'lift': 'Lift'},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 5. Correlação com o Target

In [ ]:
numeric_features = [c for c in feature_cols if df[c].dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Int16, pl.Int8)]

corr_rows = []
for c in numeric_features:
    try:
        r = df.select(pl.corr(c, 'is_dont_go_next_60m').alias('r')).item()
        corr_rows.append({'feature': c, 'pearson_r': r})
    except Exception:
        pass

corr_df = pl.DataFrame(corr_rows).with_columns(
    pl.col('pearson_r').abs().alias('abs_r')
).sort('abs_r', descending=True)

print('Top-15 features por correlação de Pearson com o target:')
print(corr_df.head(15))

In [ ]:
fig = px.bar(
    corr_df.head(20).to_pandas(),
    x='feature', y='pearson_r',
    title='Top-20 features: correlação de Pearson com is_dont_go_next_60m',
    labels={'feature': 'Feature', 'pearson_r': 'Pearson r'},
    color='pearson_r',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 6. Resumo

In [ ]:
print('=== RESUMO GOLD DATASET ===')
print(f'Registros       : {len(df):>12,}')
print(f'Features ML     : {len(feature_cols):>12,}')
print(f'  - Temporais   : {sum(1 for c in feature_cols if c in ["hora_dia","dia_semana","mes","posicao_turno","is_turno_noturno"]):>12,}')
print(f'  - Freq alarmes: {sum(1 for c in feature_cols if c.startswith("n_") or c == "aceleracao_criticos"):>12,}')
print(f'  - Fingerprint : {len(fp_cols):>12,}')
print(f'  - Equipamento : {sum(1 for c in feature_cols if c in ["frota_encoded","is_em_operacao","is_em_manutencao","sem_apontamento"]):>12,}')
print(f'\nBalanceamento target (60m):')
pos = int(df['is_dont_go_next_60m'].sum())
total = len(df)
print(f'  Positivos : {pos:,} ({100*pos/total:.3f}%)')
print(f'  Negativos : {total-pos:,} ({100*(total-pos)/total:.3f}%)')
print(f'  Ratio     : 1:{(total-pos)//max(pos,1)}')